In [1]:
import os 
import time
import requests
import pandas as pd

In [ ]:
# REST API Set-up
# EMAIL = ''

# signup_url = f'https://aqs.epa.gov/data/api/signup?email={EMAIL}'
# res = requests.get(signup_url)

# data = res.json()
# print(data)

In [ ]:
BASE_URL = 'https://aqs.epa.gov/data/api/'
AQS_EMAIL = ''
API_KEY = ''

STATE_CODE = 13

YEARS = [2023, 2024, 2025]

POLLUTANTS = {
    'PM2.5': '88101',
}

OUTPUT_DIR = 'data/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
def aqs_get(endpoint, params):
    params.update({
        'email': AQS_EMAIL,
        'key': API_KEY,
    })
    url = BASE_URL + endpoint
    res = requests.get(url, params=params, timeout=500)
    res.raise_for_status()
    data = res.json()
    header = data.get('Header', [{}])[0]
    status = header.get('status', 'unknown')
    if status != 'Success':
        print(f"API status: {status} -- {header.get('explanationOfStatus', '')}")
        return None
    body = data.get('Data', [])
    return pd.DataFrame(body)

In [ ]:
for pollutant_name, param_code in POLLUTANTS.items():
    print(f'{pollutant_name}: {param_code}')
    meta = aqs_get('monitors/byState', {
        'param': param_code,
        'bdate': f'{YEARS[0]}0101',
        'edate': f'{YEARS[-1]}1231',
        'state': STATE_CODE
    })
    if meta.empty:
        print('No monitor data found.')
    else:
        key_cols = ['poc', 'local_site_name', 'address', 'latitude', 'longitude']
        key_cols = [col for col in key_cols if col in meta.columns]
        print(meta[key_cols].to_string(index=False))
    time.sleep(5)

PM2.5: 88101
 poc  local_site_name                                                                           address  latitude  longitude
   1  Fire Station #8                     Fire Station #8, 1711 Marietta Blvd., Atlanta, Georgia, 30318 33.802241 -84.435618
   1     Sandersville                     Oconee Center, 824 School Street, Sandersville, Georgia 31082 32.967359 -82.806871
   3         Gwinnett              Gwinnett Tech, 5150 Sugarloaf Parkway, Lawrenceville, Georgia, 30043 33.963200 -84.069100
   4           Athens                                         2350 BARNETT SHOALS RD., ATHENS, GA 30605 33.918137 -83.344385
   3     Sandersville                     Oconee Center, 824 School Street, Sandersville, Georgia 31082 32.967359 -82.806871
   3    Macon-Alllied                                 300 Allied Industrial Bvld., Macon, Georgia 31206 32.777465 -83.640996
  23      Gainesville                                      695 Fair Street, Gainesville, Georgia, 30501 34.29930

In [6]:
raw_data = {}

for pollutant_name, param_code in POLLUTANTS.items():
    yearly_frames = []

    for year in YEARS:
        # Use 6-month chunks to avoid exceeding AQS server limits
        date_chunks = [
            (f'{year}0101', f'{year}0630'),
            (f'{year}0701', f'{year}1231'),
        ]

        for bdate, edate in date_chunks:
            print(f'Fetching {pollutant_name} data for {bdate} to {edate}...')
            
            try:
                df_chunk = aqs_get('dailyData/byState', {
                    'param': param_code,
                    'bdate': bdate,
                    'edate': edate,
                    'state': STATE_CODE
                })
                if df_chunk is not None and not df_chunk.empty:
                    yearly_frames.append(df_chunk)

                time.sleep(5)  # Sleep to respect API rate limits

            except Exception as e:
                print(f'Error fetching data for {bdate} to {edate}: {e}')
                continue
        
    if yearly_frames:
        raw_data[pollutant_name] = pd.concat(yearly_frames, ignore_index=True)
    else:
        print(f'No data found for {pollutant_name}.')

Fetching PM2.5 data for 20230101 to 20230630...
Fetching PM2.5 data for 20230701 to 20231231...
Fetching PM2.5 data for 20240101 to 20240630...
Fetching PM2.5 data for 20240701 to 20241231...
Fetching PM2.5 data for 20250101 to 20250630...
Fetching PM2.5 data for 20250701 to 20251231...


In [7]:
df_raw = raw_data['PM2.5']
df_raw

,state_code,county_code,site_number,parameter_code,poc,latitude,longitude,datum,parameter,sample_duration_code,...,method_code,method,local_site_name,site_address,state,county,city,cbsa_code,cbsa,date_of_last_change
0,13,121,0056,88101,1,33.778400,-84.391400,NAD83,PM2.5 - Local Conditions,7,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,NR-GA Tech,"Georgia Institute of Technology, 6th Street an...",Georgia,Fulton,Atlanta,12060,"Atlanta-Sandy Springs-Roswell, GA",2025-03-13
1,13,121,0056,88101,1,33.778400,-84.391400,NAD83,PM2.5 - Local Conditions,7,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,NR-GA Tech,"Georgia Institute of Technology, 6th Street an...",Georgia,Fulton,Atlanta,12060,"Atlanta-Sandy Springs-Roswell, GA",2025-03-13
2,13,121,0056,88101,1,33.778400,-84.391400,NAD83,PM2.5 - Local Conditions,7,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,NR-GA Tech,"Georgia Institute of Technology, 6th Street an...",Georgia,Fulton,Atlanta,12060,"Atlanta-Sandy Springs-Roswell, GA",2025-03-13
3,13,121,0056,88101,1,33.778400,-84.391400,NAD83,PM2.5 - Local Conditions,7,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,NR-GA Tech,"Georgia Institute of Technology, 6th Street an...",Georgia,Fulton,Atlanta,12060,"Atlanta-Sandy Springs-Roswell, GA",2025-03-13
4,13,121,0056,88101,1,33.778400,-84.391400,NAD83,PM2.5 - Local Conditions,7,...,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,NR-GA Tech,"Georgia Institute of Technology, 6th Street an...",Georgia,Fulton,Atlanta,12060,"Atlanta-Sandy Springs-Roswell, GA",2025-03-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315621,13,069,0002,88101,3,31.513097,-82.749971,NAD83,PM2.5 - Local Conditions,X,...,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,General Coffee,"46 John Coffee Road, Nicholls, GA 31554",Georgia,Coffee,Not in a city,20060,"Douglas, GA",2026-03-17
315622,13,069,0002,88101,3,31.513097,-82.749971,NAD83,PM2.5 - Local Conditions,X,...,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,General Coffee,"46 John Coffee Road, Nicholls, GA 31554",Georgia,Coffee,Not in a city,20060,"Douglas, GA",2026-03-17
315623,13,069,0002,88101,3,31.513097,-82.749971,NAD83,PM2.5 - Local Conditions,X,...,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,General Coffee,"46 John Coffee Road, Nicholls, GA 31554",Georgia,Coffee,Not in a city,20060,"Douglas, GA",2026-03-17
315624,13,069,0002,88101,3,31.513097,-82.749971,NAD83,PM2.5 - Local Conditions,X,...,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,General Coffee,"46 John Coffee Road, Nicholls, GA 31554",Georgia,Coffee,Not in a city,20060,"Douglas, GA",2026-03-17


In [8]:
df_raw['Date'] = pd.to_datetime(df_raw['date_local'])
df_raw['Date'] = pd.to_datetime(df_raw['Date'].astype(str), format='%Y-%m-%d')

df_raw['Year'] = df_raw['Date'].dt.year
df_raw['Month'] = df_raw['Date'].dt.month

df_raw = df_raw.sort_values('Date').reset_index(drop=True)

df_raw.to_csv(os.path.join(OUTPUT_DIR, f'{YEARS[0]}_{YEARS[-1]}_pm25_daily_data_GA.csv'), index=False)